# Feature Engineering

---

1. Import packages
2. Load data
3. Feature engineering

---

## 1. Import packages

In [1]:
import pandas as pd
import numpy as np

---
## 2. Load data

In [2]:
df = pd.read_csv("../02_Data/clean_data_after_eda.csv")
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


---

## 3. Feature engineering

### Difference between off-peak prices in December and preceding January

Below is the code created by your colleague to calculate the feature described above. Use this code to re-create this feature and then think about ways to build on this feature to create features with a higher predictive power.

In [4]:
price_df = pd.read_csv('../02_Data/price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


In [5]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


Now it is time to get creative and to conduct some of your own feature engineering! Have fun with it, explore different ideas and try to create as many as you can!

In [6]:
# Compare the customer's latest-month consumption
# with their average monthly consumption over the last 12 months

df['recent_consumption_ratio'] = (
    df['cons_last_month'] / (df['cons_12m'] / 12)
)

df[['cons_12m', 'cons_last_month', 'recent_consumption_ratio']].head()

,cons_12m,cons_last_month,recent_consumption_ratio
0,0,0,NaN
1,4660,0,0.000000
2,544,0,0.000000
3,1584,0,0.000000
4,4425,526,1.426441


In [7]:
# Difference between forecasted and historical annual consumption

df['forecast_consumption_diff'] = (
    df['forecast_cons_12m'] - df['cons_12m']
)

df[['cons_12m', 'forecast_cons_12m', 'forecast_consumption_diff']].head()

,cons_12m,forecast_cons_12m,forecast_consumption_diff
0,0,0.00,0.00
1,4660,189.95,-4470.05
2,544,47.96,-496.04
3,1584,240.04,-1343.96
4,4425,445.75,-3979.25


In [8]:
# Add Estelle's engineered price features to the customer dataset

df = pd.merge(
    df,
    diff,
    on='id',
    how='left'
)

df.head()

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,...,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn,recent_consumption_ratio,forecast_consumption_diff,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,...,99.530517,44.235794,2.086425,9.953056e+01,4.423670e+01,1,NaN,0.00,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,...,0.000000,0.000000,0.009485,1.217891e-03,0.000000e+00,0,0.000000,-4470.05,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,...,0.000000,0.000000,0.000004,9.450150e-08,0.000000e+00,0,0.000000,-496.04,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,lmkebamcaaclubfxadlmueccxoimlema,1584,0,0,2010-03-30,2016-03-30,2010-03-30,2015-03-31,240.04,...,0.000000,0.000000,0.000003,0.000000e+00,0.000000e+00,0,0.000000,-1343.96,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,MISSING,4425,0,526,2010-01-13,2016-03-07,2010-01-13,2015-03-09,445.75,...,0.000000,0.000000,0.000011,2.896760e-06,4.860000e-10,0,1.426441,-3979.25,-0.006192,0.162916


In [9]:
df[['id',
    'offpeak_diff_dec_january_energy',
    'offpeak_diff_dec_january_power']].head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,24011ae4ebbe3035111d65fa7c15bc57,0.020057,3.700961
1,d29c2c54acc38ff3c0614d0a653813dd,-0.003767,0.177779
2,764c75f661154dac3a6c254cd082ea7d,-0.004670,0.177779
3,bba03439a292a1e166f80264c16191cb,-0.004547,0.177779
4,149d57cf92fc41cf94415803a877cb4b,-0.006192,0.162916


In [10]:
# Compare forecasted consumption with historical consumption

df['forecast_consumption_ratio'] = (
    df['forecast_cons_12m'] / df['cons_12m']
)

df[['cons_12m',
    'forecast_cons_12m',
    'forecast_consumption_ratio']].head()

,cons_12m,forecast_cons_12m,forecast_consumption_ratio
0,0,0.00,NaN
1,4660,189.95,0.040762
2,544,47.96,0.088162
3,1584,240.04,0.151540
4,4425,445.75,0.100734


In [11]:
# Create a simple indicator for customers with multiple active products

df['has_multiple_products'] = (
    df['nb_prod_act'] > 1
).astype(int)

df[['nb_prod_act', 'has_multiple_products']].head()

,nb_prod_act,has_multiple_products
0,2,1
1,1,0
2,1,0
3,1,0
4,1,0


In [12]:
# Calculate the customer's contract duration in days

df['contract_duration_days'] = (
    df['date_end'] - df['date_activ']
).dt.days

df[['date_activ', 'date_end', 'contract_duration_days']].head()

,date_activ,date_end,contract_duration_days
0,2013-06-15,2016-06-15,1096
1,2009-08-21,2016-08-30,2566
2,2010-04-16,2016-04-16,2192
3,2010-03-30,2016-03-30,2192
4,2010-01-13,2016-03-07,2245


In [13]:
df[['date_activ', 'date_end', 'date_modif_prod', 'date_renewal']].describe()

,date_activ,date_end,date_modif_prod,date_renewal
count,14606,14606,14606,14606
mean,2011-01-28 07:54:18.879912448,2016-07-27 20:48:26.422018560,2013-01-02 12:29:10.951663872,2015-07-21 06:59:00.353279488
min,2003-05-09 00:00:00,2016-01-28 00:00:00,2003-05-09 00:00:00,2013-06-26 00:00:00
25%,2010-01-15 00:00:00,2016-04-27 06:00:00,2010-08-12 00:00:00,2015-04-17 00:00:00
50%,2011-03-04 00:00:00,2016-08-01 00:00:00,2013-06-19 00:00:00,2015-07-27 00:00:00
75%,2012-04-19 00:00:00,2016-10-31 00:00:00,2015-06-16 00:00:00,2015-10-29 00:00:00
max,2014-09-01 00:00:00,2017-06-13 00:00:00,2016-01-29 00:00:00,2016-01-28 00:00:00


In [14]:
# Calculate the number of days between activation and product modification

df['days_to_product_modification'] = (
    df['date_modif_prod'] - df['date_activ']
).dt.days

df[['date_activ',
    'date_modif_prod',
    'days_to_product_modification']].head()

,date_activ,date_modif_prod,days_to_product_modification
0,2013-06-15,2015-11-01,869
1,2009-08-21,2009-08-21,0
2,2010-04-16,2010-04-16,0
3,2010-03-30,2010-03-30,0
4,2010-01-13,2010-01-13,0


In [15]:
# Calculate annual consumption per active product

df['consumption_per_product'] = (
    df['cons_12m'] / df['nb_prod_act']
)

df[['cons_12m',
    'nb_prod_act',
    'consumption_per_product']].head()

,cons_12m,nb_prod_act,consumption_per_product
0,0,2,0.0
1,4660,1,4660.0
2,544,1,544.0
3,1584,1,1584.0
4,4425,1,4425.0


In [16]:
df['has_gas'].value_counts(dropna=False)

has_gas
f    11955
t     2651
Name: count, dtype: int64

In [17]:
# Convert the gas indicator into a numeric binary feature
# f = 0 (no gas), t = 1 (has gas)

df['has_gas_binary'] = (
    df['has_gas'] == 't'
).astype(int)

df[['has_gas', 'has_gas_binary']].head()

,has_gas,has_gas_binary
0,t,1
1,f,0
2,f,0
3,f,0
4,f,0


In [18]:
# Calculate net margin per active product

df['net_margin_per_product'] = (
    df['net_margin'] / df['nb_prod_act']
)

df[['net_margin',
    'nb_prod_act',
    'net_margin_per_product']].head()

,net_margin,nb_prod_act,net_margin_per_product
0,678.99,2,339.495
1,18.89,1,18.890
2,6.60,1,6.600
3,25.46,1,25.460
4,47.98,1,47.980


In [19]:
df['margin_per_power'] = (
    df['net_margin'] / df['pow_max']
)

df[['net_margin',
    'pow_max',
    'margin_per_power']].head()

,net_margin,pow_max,margin_per_power
0,678.99,43.648,15.556039
1,18.89,13.800,1.368841
2,6.60,13.856,0.476328
3,25.46,13.200,1.928788
4,47.98,19.800,2.423232


In [20]:
df['consumption_per_power'] = (
    df['cons_12m'] / df['pow_max']
)

df[['cons_12m',
    'pow_max',
    'consumption_per_power']].head()

,cons_12m,pow_max,consumption_per_power
0,0,43.648,0.000000
1,4660,13.800,337.681159
2,544,13.856,39.260970
3,1584,13.200,120.000000
4,4425,19.800,223.484848


In [21]:
df['forecast_vs_actual_pct'] = (
    (df['forecast_cons_12m'] - df['cons_12m'])
    / df['cons_12m']
) * 100

df[['forecast_cons_12m',
    'cons_12m',
    'forecast_vs_actual_pct']].head()

,forecast_cons_12m,cons_12m,forecast_vs_actual_pct
0,0.00,0,NaN
1,189.95,4660,-95.923820
2,47.96,544,-91.183824
3,240.04,1584,-84.845960
4,445.75,4425,-89.926554


In [22]:
df.shape

(14606, 58)

In [23]:
df.isna().sum().sort_values(ascending=False).head(20)

recent_consumption_ratio      117
forecast_consumption_ratio    100
forecast_vs_actual_pct        100
margin_per_power                0
net_margin_per_product          0
var_year_price_off_peak         0
var_year_price_peak             0
var_year_price_mid_peak         0
var_6m_price_off_peak_var       0
var_6m_price_peak_var           0
var_6m_price_mid_peak_var       0
var_6m_price_off_peak_fix       0
var_6m_price_peak_fix           0
var_6m_price_mid_peak_fix       0
var_6m_price_off_peak           0
var_6m_price_peak               0
var_6m_price_mid_peak           0
churn                           0
channel_sales                   0
forecast_consumption_diff       0
dtype: int64

In [24]:
df.select_dtypes(include=np.number).isin([np.inf, -np.inf]).sum().sort_values(ascending=False).head(20)

forecast_vs_actual_pct             17
forecast_consumption_ratio         17
forecast_consumption_diff           0
var_6m_price_peak_var               0
var_6m_price_mid_peak_var           0
var_6m_price_off_peak_fix           0
var_6m_price_peak_fix               0
var_6m_price_mid_peak_fix           0
var_6m_price_off_peak               0
var_6m_price_peak                   0
var_6m_price_mid_peak               0
churn                               0
recent_consumption_ratio            0
offpeak_diff_dec_january_energy     0
cons_gas_12m                        0
offpeak_diff_dec_january_power      0
has_multiple_products               0
contract_duration_days              0
days_to_product_modification        0
consumption_per_product             0
dtype: int64

In [25]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [26]:
df.select_dtypes(include=np.number).isin([np.inf, -np.inf]).sum().sum()

np.int64(0)

In [27]:
df.columns.tolist()

['id',
 'channel_sales',
 'cons_12m',
 'cons_gas_12m',
 'cons_last_month',
 'date_activ',
 'date_end',
 'date_modif_prod',
 'date_renewal',
 'forecast_cons_12m',
 'forecast_cons_year',
 'forecast_discount_energy',
 'forecast_meter_rent_12m',
 'forecast_price_energy_off_peak',
 'forecast_price_energy_peak',
 'forecast_price_pow_off_peak',
 'has_gas',
 'imp_cons',
 'margin_gross_pow_ele',
 'margin_net_pow_ele',
 'nb_prod_act',
 'net_margin',
 'num_years_antig',
 'origin_up',
 'pow_max',
 'var_year_price_off_peak_var',
 'var_year_price_peak_var',
 'var_year_price_mid_peak_var',
 'var_year_price_off_peak_fix',
 'var_year_price_peak_fix',
 'var_year_price_mid_peak_fix',
 'var_year_price_off_peak',
 'var_year_price_peak',
 'var_year_price_mid_peak',
 'var_6m_price_off_peak_var',
 'var_6m_price_peak_var',
 'var_6m_price_mid_peak_var',
 'var_6m_price_off_peak_fix',
 'var_6m_price_peak_fix',
 'var_6m_price_mid_peak_fix',
 'var_6m_price_off_peak',
 'var_6m_price_peak',
 'var_6m_price_mid_peak',


In [28]:
df.dtypes

id                                         object
channel_sales                              object
cons_12m                                    int64
cons_gas_12m                                int64
cons_last_month                             int64
date_activ                         datetime64[ns]
date_end                           datetime64[ns]
date_modif_prod                    datetime64[ns]
date_renewal                       datetime64[ns]
forecast_cons_12m                         float64
forecast_cons_year                          int64
forecast_discount_energy                  float64
forecast_meter_rent_12m                   float64
forecast_price_energy_off_peak            float64
forecast_price_energy_peak                float64
forecast_price_pow_off_peak               float64
has_gas                                    object
imp_cons                                  float64
margin_gross_pow_ele                      float64
margin_net_pow_ele                        float64


In [29]:
engineered_features = [
    'recent_consumption_ratio',
    'forecast_consumption_diff',
    'offpeak_diff_dec_january_energy',
    'offpeak_diff_dec_january_power',
    'forecast_consumption_ratio',
    'has_multiple_products',
    'contract_duration_days',
    'days_to_product_modification',
    'consumption_per_product',
    'has_gas_binary',
    'net_margin_per_product',
    'margin_per_power',
    'consumption_per_power',
    'forecast_vs_actual_pct'
]

df[engineered_features].describe().T

,count,mean,std,min,25%,50%,75%,max
recent_consumption_ratio,14489.0,0.926510,1.027173,0.000000e+00,0.000000,0.873417,1.352519,1.531579e+01
forecast_consumption_diff,14606.0,-157351.671373,573006.989026,-6.207019e+06,-37462.037500,-12605.765000,-4961.927500,3.534240e+03
offpeak_diff_dec_january_energy,14606.0,-0.004566,0.012049,-1.484770e-01,-0.007829,-0.005344,-0.003520,1.689580e-01
offpeak_diff_dec_january_power,14606.0,0.278495,1.349231,-4.426693e+01,0.000004,0.162916,0.177779,4.072888e+01
forecast_consumption_ratio,14489.0,0.093871,0.054776,0.000000e+00,0.044917,0.102448,0.147502,6.246222e-01
has_multiple_products,14606.0,0.217376,0.412475,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00
contract_duration_days,14606.0,2007.537587,604.875654,7.310000e+02,1461.000000,1828.500000,2353.000000,4.795000e+03
days_to_product_modification,14606.0,705.190880,844.180115,-5.700000e+01,0.000000,32.000000,1398.000000,4.367000e+03
consumption_per_product,14606.0,118587.307547,408239.729509,0.000000e+00,4658.166667,11950.000000,35121.083333,5.731448e+06
has_gas_binary,14606.0,0.181501,0.385446,0.000000e+00,0.000000,0.000000,0.000000,1.000000e+00


In [30]:
df.loc[
    df['days_to_product_modification'] < 0,
    ['date_activ', 'date_modif_prod', 'days_to_product_modification']
].head(10)

,date_activ,date_modif_prod,days_to_product_modification
2930,2011-05-14,2011-05-03,-11
3261,2013-05-14,2013-03-18,-57
4828,2010-03-26,2010-03-24,-2
4851,2012-04-02,2012-03-30,-3
5018,2012-03-13,2012-03-10,-3
5048,2005-05-13,2005-04-29,-14
5300,2013-06-10,2013-05-28,-13
5398,2010-09-15,2010-09-13,-2
5829,2010-06-22,2010-06-18,-4
6256,2013-05-07,2013-03-18,-50


In [31]:
df[
    ['recent_consumption_ratio',
     'forecast_consumption_ratio',
     'forecast_vs_actual_pct']
].isna().sum()

recent_consumption_ratio      117
forecast_consumption_ratio    117
forecast_vs_actual_pct        117
dtype: int64

In [32]:
df['zero_annual_consumption'] = (
    df['cons_12m'] == 0
).astype(int)

df[['cons_12m',
    'zero_annual_consumption']].head()

,cons_12m,zero_annual_consumption
0,0,1
1,4660,0
2,544,0
3,1584,0
4,4425,0


In [33]:
df['zero_annual_consumption'].value_counts()

zero_annual_consumption
0    14489
1      117
Name: count, dtype: int64

In [34]:
id="8x9q2m"
df[engineered_features + ['zero_annual_consumption']].corr()

,recent_consumption_ratio,forecast_consumption_diff,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power,forecast_consumption_ratio,has_multiple_products,contract_duration_days,days_to_product_modification,consumption_per_product,has_gas_binary,net_margin_per_product,margin_per_power,consumption_per_power,forecast_vs_actual_pct,zero_annual_consumption
recent_consumption_ratio,1.000000,-0.077546,-0.014456,-0.003813,-0.094272,0.020407,0.016705,0.002747,0.073710,0.021439,0.031629,0.021665,0.072913,-0.094272,NaN
forecast_consumption_diff,-0.077546,1.000000,0.018741,0.019736,0.418218,-0.182497,0.024697,0.139851,-0.929516,-0.212735,-0.104563,-0.171497,-0.920066,0.418218,0.024709
offpeak_diff_dec_january_energy,-0.014456,0.018741,1.000000,0.468083,0.103018,0.103282,-0.036753,-0.017166,-0.020802,-0.003289,-0.027781,0.049925,-0.002646,0.103018,0.009095
offpeak_diff_dec_january_power,-0.003813,0.019736,0.468083,1.000000,0.051224,0.009722,-0.000658,-0.004805,-0.021075,-0.013287,-0.060320,-0.050460,-0.014427,0.051224,0.010626
forecast_consumption_ratio,-0.094272,0.418218,0.103018,0.051224,1.000000,-0.054602,-0.068353,0.118413,-0.428110,-0.076310,-0.066111,-0.028213,-0.395075,1.000000,NaN
has_multiple_products,0.020407,-0.182497,0.103282,0.009722,-0.054602,1.000000,-0.003812,0.018844,0.067812,0.893512,-0.125524,0.082225,0.166462,-0.054602,-0.004531
contract_duration_days,0.016705,0.024697,-0.036753,-0.000658,-0.068353,-0.003812,1.000000,0.240280,-0.029020,0.007925,-0.008477,-0.038064,-0.018103,-0.068353,0.059233
days_to_product_modification,0.002747,0.139851,-0.017166,-0.004805,0.118413,0.018844,0.240280,1.000000,-0.138558,-0.007253,-0.005257,-0.023589,-0.138843,0.118413,-0.027734
consumption_per_product,0.073710,-0.929516,-0.020802,-0.021075,-0.428110,0.067812,-0.029020,-0.138558,1.000000,0.095220,0.150046,0.180184,0.858787,-0.428110,-0.026104
has_gas_binary,0.021439,-0.212735,-0.003289,-0.013287,-0.076310,0.893512,0.007925,-0.007253,0.095220,1.000000,-0.101098,0.088764,0.193812,-0.076310,-0.006448


## Feature Engineering – Final Interpretation & Conclusion

Feature engineering was performed to transform the raw customer and pricing information into more meaningful variables for churn analysis.

The engineered features captured important business dimensions such as:

- **Consumption behaviour:** recent consumption, forecast vs actual consumption, consumption per product, and consumption relative to subscribed power.
- **Price sensitivity:** changes in off-peak energy and power prices between January and December.
- **Customer characteristics:** multiple products and gas subscription.
- **Profitability:** net margin per product and margin relative to subscribed power.
- **Customer timeline:** contract duration and time between activation and product modification.
- **Zero-consumption behaviour:** an indicator was created for customers with zero annual consumption.

During validation, ratio-based features produced missing values where the denominator was zero, and infinite values were converted to `NaN`. These cases were retained for appropriate handling during the later preprocessing stage rather than being arbitrarily replaced.

Some engineered variables showed strong skewness and extreme values. These were not automatically removed because they may represent genuine differences between customers.

### Conclusion

Feature engineering successfully enriched the dataset with business-driven features that provide additional information about customer behaviour, pricing, consumption, profitability, products, and customer timelines.

The final dataset contains **14,606 customers and 59 columns** and is now ready for the next stage of the project.

> **Key takeaway:** Feature engineering transformed raw customer and pricing data into meaningful variables that can provide richer signals for predicting customer churn.